<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_10_Fine_Tuning_Domain_Adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
domain_texts = [
    "Artificial Intelligence in healthcare uses machine learning algorithms to assist with medical data analysis.",

    "Machine learning can help healthcare professionals identify patterns in medical datasets.",

    "Deep learning is used in medical imaging to analyze images such as X-rays, CT scans and MRI scans.",

    "Natural Language Processing can help healthcare systems process clinical notes and medical documents.",

    "AI-based healthcare systems can support early detection of diseases by identifying patterns in patient data.",

    "Computer vision techniques can be used to analyze medical images and assist doctors in identifying abnormalities.",

    "Predictive models in healthcare can analyze historical patient data to estimate potential health risks.",

    "Generative AI can assist healthcare professionals by summarizing medical information and generating structured reports.",

    "AI systems in healthcare should be carefully evaluated because incorrect predictions can affect medical decisions.",

    "Data privacy and security are important considerations when developing AI systems for healthcare."
]

print("Number of training examples:", len(domain_texts))

Number of training examples: 10


In [4]:
dataset = Dataset.from_dict({
    "text": domain_texts
})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 10
})


In [5]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Pre-trained model loaded successfully!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Pre-trained model loaded successfully!


In [6]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print("Padding token configured successfully!")

Padding token configured successfully!


In [7]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenized successfully!")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset tokenized successfully!


In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created successfully!")

Data collator created successfully!


In [10]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_healthcare_model",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_strategy="no",
    logging_steps=1,
    report_to="none"
)

print("Training configuration created successfully!")

Training configuration created successfully!


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [12]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.974886
2,4.839794
3,3.756352
4,3.950312
5,4.794572
6,3.463149
7,3.416536
8,3.146032
9,3.516189
10,2.955837


TrainOutput(global_step=10, training_loss=3.781365919113159, metrics={'train_runtime': 39.769, 'train_samples_per_second': 0.503, 'train_steps_per_second': 0.251, 'total_flos': 653241876480.0, 'train_loss': 3.781365919113159, 'epoch': 2.0})